# Projet IC2 (2/3) — Modèles et validation temporelle

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ic2/notebooks/IC2_Projet_S4_modeles.ipynb)
> Données : dossier `IC2_TP_donnees`. Bibliothèques : pandas, numpy, matplotlib, scikit-learn, torch, shap.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['denoyage_fosse.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ic2/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error
import torch, torch.nn as nn
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
pit = pd.read_csv('denoyage_fosse.csv', parse_dates=['semaine'])
f = pd.DataFrame(index=pit.index)
for c in ['P1_fosse', 'P2_150m', 'P3_600m']:
    for lag in (0, 1, 2, 4): f[f'{c}_l{lag}'] = pit[c].shift(lag)
f['pluie_4s'] = pit['pluie_mm'].rolling(4).sum(); f['pluie_12s'] = pit['pluie_mm'].rolling(12).sum()
f['pompage_t'] = pit['pompage_m3h']; f['pompage_plan_t4'] = pit['pompage_m3h'].shift(-4)
j = pit['semaine'].dt.dayofyear; f['sin_saison'] = np.sin(2*np.pi*j/365); f['cos_saison'] = np.cos(2*np.pi*j/365)
f['cible'] = pit['P1_fosse'].shift(-4); f['delta'] = f['cible'] - pit['P1_fosse']
f = f.dropna()
date = pit.loc[f.index, 'semaine']; tr, te = f[date < '2024-01-01'], f[date >= '2024-01-01']
X = [c for c in f.columns if c not in ('cible', 'delta')]
rmse = lambda a, b: np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2))
print(len(tr), 'semaines d’entraînement,', len(te), 'de test')

## 1. Baselines et modèles tabulaires

In [ ]:
res = {'persistance': rmse(te['cible'], te['P1_fosse_l0'])}
gb_abs = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=15, random_state=0).fit(tr[X], tr['cible'])
res['GBM niveau absolu'] = rmse(te['cible'], gb_abs.predict(te[X]))
gb = HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_leaf_nodes=15, random_state=0).fit(tr[X], tr['delta'])
res['GBM delta'] = rmse(te['cible'], te['P1_fosse_l0'] + gb.predict(te[X]))
rd = make_pipeline(StandardScaler(), Ridge(1.0)).fit(tr[X], tr['delta'])
res['Ridge delta'] = rmse(te['cible'], te['P1_fosse_l0'] + rd.predict(te[X]))
print({k: round(v, 2) for k, v in res.items()})

## 2. LSTM sur 12 semaines + pompage planifié

In [ ]:
cols = ['P1_fosse', 'P2_150m', 'P3_600m', 'pluie_mm', 'pompage_m3h']; Lw = 12; split = tr.index.max() + 1
raw = pit[cols].values.astype(np.float32); mu, sd = raw[:split].mean(0), raw[:split].std(0); rn = (raw - mu) / sd
idx = np.array([i for i in f.index if i >= Lw - 1]); Xs = np.stack([rn[i - Lw + 1:i + 1] for i in idx])
fut = ((pit['pompage_m3h'].shift(-4).values[idx] - mu[4]) / sd[4]).astype(np.float32)[:, None]
dsd = tr['delta'].std(); ys = (f.loc[idx, 'delta'].values / dsd).astype(np.float32); itr = idx < split
class LSTMf(nn.Module):
    def __init__(s): super().__init__(); s.l = nn.LSTM(5, 32, batch_first=True); s.o = nn.Sequential(nn.Linear(33, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(s, x, u): h, _ = s.l(x); return s.o(torch.cat([h[:, -1], u], 1))
torch.manual_seed(0); lm = LSTMf(); opt = torch.optim.Adam(lm.parameters(), 3e-3)
Xt, Ut, Yt = torch.tensor(Xs[itr]), torch.tensor(fut[itr]), torch.tensor(ys[itr])[:, None]
for ep in range(400):
    opt.zero_grad(); l = nn.functional.mse_loss(lm(Xt, Ut), Yt); l.backward(); opt.step()
with torch.no_grad(): p = lm(torch.tensor(Xs[~itr]), torch.tensor(fut[~itr])).numpy().ravel() * dsd + f.loc[idx[~itr], 'P1_fosse_l0'].values
res['LSTM delta'] = rmse(f.loc[idx[~itr], 'cible'], p); print({k: round(v, 2) for k, v in res.items()})

## Discussion
Pourquoi la Ridge sur Δh gagne-t-elle ? Pourquoi le GBM sur le niveau absolu échoue-t-il en 2024–2025 ? Proposez un modèle physique simple comme baseline supplémentaire.